# Разведочный анализ PM2.5 (EDA)

**Цель.** Увидеть закономерности, которые потом проверим тестами (H1 — сезон, H2 — погода, H3 — суточный цикл), и оценить масштаб превышений нормы ВОЗ.

**Входы:** `data/processed/hourly.csv` (часы) и `data/processed/daily.csv` (сутки).
**Выходы:** графики в `figures/` (`00_pm25_distribution.png` и далее).

EDA — это разведка: мы смотрим и описываем, но не доказываем. Выводы подтвердим тестами в задаче 5.1, чтобы не «ловить звёздочки» на графиках.

**Зерно данных.** Суточные графики и статистика — по таблице `daily` (только сутки со средним `pm25_mean`); суточный цикл — по часовой таблице `hourly` (только часы со значением).

In [1]:
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.append("..")
from src import clean, eda, features

HOURLY_PATH = "../data/processed/hourly.csv"
DAILY_PATH = "../data/processed/daily.csv"
FIGURES_DIR = "../figures"

WHO_LIMIT = features.WHO_DAILY_LIMIT
SEASON_ORDER = ["winter", "spring", "summer", "autumn"]
HEATING_LABELS = {True: "Отопительный сезон", False: "Вне сезона"}

# Единая тема графиков: палитра, удобная для дальтоников
sns.set_theme(style="whitegrid", context="talk", palette="colorblind")

## Шаг 1. Читаем таблицы и проверяем контракты

Читаем готовые таблицы функциями проекта и сразу проверяем их `check_*`: если контракт нарушен, дальше смотреть нечего. Суточные графики строим только по суткам, где есть среднее PM2.5, поэтому выделяем их в `days`.

In [2]:
hourly = clean.read_clean_csv(HOURLY_PATH)
features.check_hourly(hourly)
daily = features.read_daily_csv(DAILY_PATH)
features.check_daily(daily)

print("hourly:", hourly.shape)
print("daily:", daily.shape)

# Сутки со средним — с ними работают суточные графики
days = daily[daily["pm25_mean"].notna()]
print("суток со средним PM2.5:", len(days))

hourly: (49075, 18)
daily: (2044, 16)
суток со средним PM2.5: 1534


## Шаг 2. Сколько у нас данных

Средние и медианы pandas считает по непустым значениям. Мы не заполняем пропуски заново: короткие дыры уже заполнены на этапе очистки, длинные остаются честно пустыми. Поэтому важно знать, сколько значений стоит за каждым числом: сколько часов и суток пустые и как сутки со средним распределены по годам и месяцам.

In [3]:
hours_missing = hourly["pm25"].isna().sum()
print("часов без pm25:", hours_missing, f"({hours_missing / len(hourly):.2%})")
print("часов, заполненных интерполяцией:", hourly["pm25_interpolated"].sum())
print("суток со средним:", len(days), "| без среднего:", len(daily) - len(days))

# Сколько суток со средним в каждом году и месяце
days_by_year_month = days.assign(year=days["date"].dt.year).pivot_table(
    values="pm25_mean", index="year", columns="month", aggfunc="count"
)
print(days_by_year_month)

часов без pm25: 10235 (20.86%)
часов, заполненных интерполяцией: 1148
суток со средним: 1534 | без среднего: 510
month    1     2     3     4     5     6     7     8     9     10    11    12
year                                                                         
2020    NaN   NaN   NaN  17.0  20.0  17.0  18.0  22.0  24.0  17.0  23.0  23.0
2021   21.0  18.0  22.0  18.0  12.0   9.0   6.0  17.0  14.0   9.0  23.0  27.0
2022   20.0  26.0  26.0  26.0   NaN   NaN   NaN   NaN  22.0  28.0  28.0  20.0
2023   24.0   2.0   9.0  26.0  31.0  30.0  29.0  31.0  25.0  31.0  30.0  31.0
2024   26.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  26.0  22.0  30.0  31.0
2025   31.0  28.0  31.0  30.0  31.0  30.0  31.0  31.0  28.0  31.0  13.0   NaN


**Что видим.** (вывод шага 2) Без `pm25` 10 235 часов из 49 075 (20.86 %), ещё 1 148 часов заполнены интерполяцией на этапе очистки. Суток со средним 1 534, без среднего 510. Главная дыра — май–август 2022: в таблице «год × месяц» там пусто, а в феврале–марте 2023 и ноябре 2025 мало суток (2, 9 и 13). Поэтому месячные статистики объединяют разное число лет, и это надо помнить при выводах.

## Шаг 3. Описательная статистика PM2.5

Считаем основные числа: среднее, медиану, квартили. PM2.5 скошен вправо (редкие пики смога), поэтому главные числа — медиана и межквартильный размах; среднее тянется за пиками. Сравниваем сезоны, отопительный сезон и месяцы.

In [4]:
print("Сутки, pm25_mean:")
print(days["pm25_mean"].describe())
print()
print("Часы, pm25:")
print(hourly["pm25"].describe())

Сутки, pm25_mean:
count    1534.000000
mean       29.065487
std        30.420802
min         3.289474
25%        11.010417
50%        16.871711
75%        32.526042
max       229.000000
Name: pm25_mean, dtype: float64

Часы, pm25:
count    38840.000000
mean        29.480947
std         37.746807
min          0.000000
25%          9.000000
50%         15.000000
75%         31.000000
max        421.000000
Name: pm25, dtype: float64


In [5]:
print("По сезонам (суточное среднее):")
print(days.groupby("season")["pm25_mean"].describe().reindex(SEASON_ORDER))
print()
print("По отопительному сезону:")
print(days.groupby("heating_season")["pm25_mean"].describe())

По сезонам (суточное среднее):
        count       mean        std        min        25%        50%  \
season                                                                 
winter  356.0  65.285704  40.476166  10.020833  32.757292  57.291667   
spring  391.0  17.597151  10.317504   4.021739  10.770833  14.375000   
summer  363.0  10.775799   3.751422   3.289474   8.458333  10.333333   
autumn  424.0  24.888314  18.397583   4.916667  13.875000  19.302083   

              75%         max  
season                         
winter  83.156250  229.000000  
spring  21.572917   79.416667  
summer  12.375000   33.625000  
autumn  29.489583  122.000000  

По отопительному сезону:
                count       mean        std       min        25%        50%  \
heating_season                                                                
False           763.0  12.617389   5.241042  3.289474   9.048611  11.375000   
True            771.0  45.342918  35.803281  4.500000  19.750000  32.208333   

 

In [6]:
print("По месяцам:")
print(days.groupby("month")["pm25_mean"].describe())

По месяцам:
       count       mean        std        min        25%        50%  \
month                                                                 
1      122.0  66.969343  43.870727  10.020833  32.354167  56.500000   
2      102.0  62.189044  29.424741  13.250000  46.223958  58.250000   
3      119.0  25.647255  13.094606   5.583333  16.687500  23.333333   
4      147.0  16.349315   6.595765   4.076389  11.458333  14.825000   
5      125.0  11.400908   4.203654   4.021739   8.583333  11.000000   
6      116.0  11.637772   4.418271   4.111111   8.645833  10.937500   
7      115.0  10.128128   3.414945   3.289474   7.979167   9.666667   
8      132.0  10.582567   3.246550   5.000000   8.657895  10.479167   
9      139.0  14.694215   5.239754   4.916667  10.718750  14.333333   
10     138.0  21.461171   8.767164   6.333333  15.062500  20.458333   
11     147.0  37.744951  24.619075   6.083333  20.354167  31.416667   
12     132.0  66.122487  44.538203  11.583333  29.934896  57.4583

In [7]:
# Скошенность: среднее больше медианы и положительный skew — длинный правый хвост
print("суточный pm25_mean: среднее", round(days["pm25_mean"].mean(), 1),
      "| медиана", round(days["pm25_mean"].median(), 1),
      "| skew", round(days["pm25_mean"].skew(), 2))
print("часовой pm25: среднее", round(hourly["pm25"].mean(), 1),
      "| медиана", round(hourly["pm25"].median(), 1),
      "| skew", round(hourly["pm25"].skew(), 2))

суточный pm25_mean: среднее 29.1 | медиана 16.9 | skew 2.48
часовой pm25: среднее 29.5 | медиана 15.0 | skew 2.91


**Что видим.** (вывод шага 3) Распределение скошено вправо: среднее суточного PM2.5 29.1 µg/m³ больше медианы 16.9, skew 2.48 (у часовых данных 29.5, 15.0 и 2.91). По сезонам медиана зимой 57.3 µg/m³, весной 14.4, летом 10.3, осенью 19.3. Отопительный сезон: медиана 32.2 µg/m³ (771 сутки), вне сезона 11.4 (763 суток); максимум суток 229 µg/m³. По месяцам самое высокое среднее в январе (67.0 µg/m³), самая высокая медиана в феврале (58.3), самые чистые месяцы — июль и август (медианы 9.7 и 10.5). Это описание; различие сезонов проверим тестом в 5.1.

## Шаг 4. Распределение: почему дальше Mann–Whitney

Слева — гистограмма суточного PM2.5 с линией нормы ВОЗ. Справа — ECDF (эмпирическая функция распределения): точка (x, y) значит «в доле y дней PM2.5 не превышал x»; кривая правее — значения выше. Две кривые — отопительный сезон и остальное время. ECDF сравнивает распределения целиком и не зависит от числа столбцов гистограммы. Если распределение скошено, сравнивать сезоны лучше непараметрическим тестом Mann–Whitney, а не t-тестом.

In [8]:
mean_pm25 = days["pm25_mean"].mean()
median_pm25 = days["pm25_mean"].median()
skew_pm25 = days["pm25_mean"].skew()
print(f"среднее {mean_pm25:.1f}, медиана {median_pm25:.1f}, skew {skew_pm25:.2f}")

fig, (ax_hist, ax_ecdf) = plt.subplots(1, 2, figsize=(16, 6))

sns.histplot(days["pm25_mean"], bins=50, ax=ax_hist)
ax_hist.axvline(WHO_LIMIT, color="black", linestyle="--", label="Норма ВОЗ 15 µg/m³")
ax_hist.set_xlabel("PM2.5, µg/m³")
ax_hist.set_ylabel("Число дней")
ax_hist.legend()

sns.ecdfplot(data=days, x="pm25_mean", hue=days["heating_season"].map(HEATING_LABELS), ax=ax_ecdf)
ax_ecdf.axvline(WHO_LIMIT, color="black", linestyle="--")
ax_ecdf.set_xlabel("PM2.5, µg/m³")
ax_ecdf.set_ylabel("Доля дней с PM2.5 ≤ x")
ax_ecdf.get_legend().set_title("Период")

fig.suptitle(f"PM2.5 скошен вправо: среднее {mean_pm25:.0f} больше медианы {median_pm25:.0f} µg/m³")
fig.savefig(f"{FIGURES_DIR}/00_pm25_distribution.png", dpi=150, bbox_inches="tight")
plt.close(fig)

среднее 29.1, медиана 16.9, skew 2.48


**Что видим.** (вывод шага 4) Гистограмма показывает длинный правый хвост: среднее 29.1 µg/m³ больше медианы 16.9, skew 2.48. Кривая ECDF отопительного сезона идёт правее кривой остального времени — в отопительный сезон значения выше. Раз распределение скошено, для сравнения сезонов в 5.1 берём непараметрический тест Mann–Whitney.

![График 0](../figures/00_pm25_distribution.png)

## Итог (часть 1)

(будет дополнен после графиков 1–7)